# MoMTSim-KAN — Pipeline complet

Réimplémentation NumPy/PyTorch/Pandas/Plotly de MoMTSim, adaptée aux 5 scénarios de fraude formalisés au Chapitre 3 du mémoire (ATO, Refund Fraud, Fake Credentials, Split Deposit, Smurfing), suivie de l'ingénierie des features (section 3.2.6 / 3.3) et de la validation topologique KAN (section 4.1).

**Structure du notebook :**
1. Imports & configuration globale
2. Config des 5 scénarios de fraude (JSON en dur)
3. Chargement des paramètres (`TorchParameters`)
4. Moteur de simulation vectorisé (`TorchMoMTSimEngine`)
5. Injecteur de fraude (`TorchFraudInjector`) — 5 scénarios, formules section 3.2
6. Calibration SSE / SPSA (section 3.1.3)
7. Exécution du pipeline de génération (calibration → simulation complète)
8. Feature engineering vectorisé (section 3.2.6, 12 features)
9. Validation topologique KAN (section 4.1 — normalisation, PCA, Fisher, KS)
10. Visualisations Plotly

**Prérequis de fichiers (à placer dans `./paramFiles/`) :**
`transactionsTypes.csv`, `clientsProfiles.csv`, `aggregatedTransactions.csv`, `initialBalancesDistribution.csv`, `overdraftLimits.csv`, `maxOccurrencesPerClient.csv`

**Stack : NumPy, PyTorch, Pandas, Plotly uniquement.** Aucune autre dépendance externe (json est stdlib).

## 1. Imports & configuration globale

In [1]:
from __future__ import annotations
import numpy as np
import pandas as pd
import torch
import json
import os
import math
import plotly.graph_objects as go
from plotly.subplots import make_subplots

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 1000
PARAM_DIR = "./paramFiles"
FRAUD_CONFIG_PATH = "./fraudScenariosConfig.json"

print(f"Device utilisé : {DEVICE}")

Device utilisé : cpu


## 2. Config des 5 scénarios de fraude

Écrit le fichier `fraudScenariosConfig.json` attendu par `TorchParameters`. Valeurs reprises du mémoire (section 3.2) ; deux hypothèses non sourcées sont signalées en commentaire (`S_seuil`, `tariff_grid`).

In [ ]:
FRAUD_CONFIG = {
    "global": {
        "fraud_target_min": 0.20,
        "fraud_target_max": 0.26,
        "scenarios_equal_share": True
    },
    "ato": {  # 3.2.1 — Account Takeover
        "B_min": 50000,
        "n_min": 2,
        "n_max": 6,
        "frag_min": 0.15,
        "frag_max": 0.35,
        "lambda_ato": 6.0
    },
    "refund": {  # 3.2.2 — Refund Fraud
        "p_refund_threshold": 0.7,
        "delay_min_hours": 1,
        "delay_max_hours": 48,
        "k_max": 4,
        "ratio_legit": 0.30
    },
    "fake_credentials": {  # 3.2.3 — Fake Credentials
        "dormance_min_days": 7,
        "dormance_max_days": 30,
        "n_leg_min": 3,
        "n_leg_max": 8,
        "m_leg_max": 5000,
        "m_exp_ratio_min": 0.5
    },
    "split_deposit": {  # 3.2.4 — Split Deposit
        "epsilon_max": 500,
        "T_split_min_sec": 60,
        "T_split_max_sec": 120,
        # Grille tarifaire hypothétique (aucun barème MTN/Orange public cité dans le mémoire)
        "tariff_grid": [
            {"threshold": 0, "commission": 0},
            {"threshold": 1000, "commission": 50},
            {"threshold": 5000, "commission": 150},
            {"threshold": 20000, "commission": 400},
            {"threshold": 50000, "commission": 750},
            {"threshold": 100000, "commission": 1200}
        ]
    },
    "smurfing": {  # 3.2.5 — Smurfing (Zhdanova et al.)
        "n_mules_min": 4,
        "n_mules_max": 10,
        "pct_conscious": 0.60,
        "pct_unconscious": 0.40,
        # Seuil de déclaration COBAC : hypothèse non sourcée explicitement dans le mémoire, à valider
        "S_seuil": 500000,
        "delta_min": 0.01,
        "delta_max": 0.10,
        "delay_mule_min_hours": 2,
        "delay_mule_max_hours": 24,
        "operation_interval_days": 30,
        "n_leg_mule_min": 5,
        "n_leg_mule_max": 15
    }
}

with open(FRAUD_CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(FRAUD_CONFIG, f, indent=2)

print(f"Config fraude écrite dans {FRAUD_CONFIG_PATH}")

## 3. `TorchParameters` — chargement des CSV et pré-calcul des tenseurs clients

In [2]:
class TorchParameters:
    """Charge les 6 CSV + config fraude JSON, pré-calcule les tenseurs par client
    (profil moyen/std par action tiré une fois à l'initialisation)."""

    ACTIONS = ["CASH_IN", "CASH_OUT", "DEBIT", "PAYMENT", "TRANSFER", "DEPOSIT"]
    IN_ACTIONS = {"CASH_IN", "DEPOSIT"}
    OUT_ACTIONS = {"CASH_OUT", "DEBIT", "PAYMENT", "TRANSFER"}

    def __init__(self, param_dir: str, fraud_config_path: str, n_clients: int, seed: int = 1000):
        self.rng = np.random.default_rng(seed)
        self.n_clients = n_clients

        self.client_profiles_df = pd.read_csv(f"{param_dir}/clientsProfiles.csv")
        self.agg_df = pd.read_csv(f"{param_dir}/aggregatedTransactions.csv")
        self.balances_df = pd.read_csv(f"{param_dir}/initialBalancesDistribution.csv")
        self.overdraft_df = pd.read_csv(f"{param_dir}/overdraftLimits.csv")
        self.max_occ_df = pd.read_csv(f"{param_dir}/maxOccurrencesPerClient.csv")

        with open(fraud_config_path, "r", encoding="utf-8") as f:
            self.fraud_config = json.load(f)

        self._build_client_tensors()
        self._build_step_target_tensors()

    def _sample_profile_pool(self, action: str):
        sub = self.client_profiles_df[self.client_profiles_df["action"] == action]
        if sub.empty:
            return (np.zeros(self.n_clients), np.zeros(self.n_clients),
                    np.zeros(self.n_clients, dtype=bool))
        weights = sub["weight"].values / sub["weight"].values.sum()
        idx = self.rng.choice(len(sub), size=self.n_clients, p=weights)
        means = sub["mean_amount"].values[idx]
        stds = sub["std_amount"].values[idx]
        return means, stds, np.ones(self.n_clients, dtype=bool)

    def _build_client_tensors(self):
        n = self.n_clients
        mean_mat = np.zeros((n, len(self.ACTIONS)))
        std_mat = np.zeros((n, len(self.ACTIONS)))
        avail_mat = np.zeros((n, len(self.ACTIONS)), dtype=bool)

        for j, action in enumerate(self.ACTIONS):
            means, stds, avail = self._sample_profile_pool(action)
            mean_mat[:, j] = means
            std_mat[:, j] = stds
            avail_mat[:, j] = avail

        self.mean_amount_tensor = torch.tensor(mean_mat, dtype=torch.float32, device=DEVICE)
        self.std_amount_tensor = torch.tensor(std_mat, dtype=torch.float32, device=DEVICE)
        self.action_available = torch.tensor(avail_mat, dtype=torch.bool, device=DEVICE)

        lo = self.balances_df["range_min"].values
        hi = self.balances_df["range_max"].values
        props = self.balances_df["proportion"].values / self.balances_df["proportion"].values.sum()
        bin_idx = self.rng.choice(len(self.balances_df), size=n, p=props)
        init_balance = self.rng.uniform(lo[bin_idx], hi[bin_idx])
        self.initial_balance = torch.tensor(init_balance, dtype=torch.float32, device=DEVICE)

        mean_global = mean_mat[np.arange(n)[:, None], np.arange(len(self.ACTIONS))].mean(axis=1)
        overdraft = np.zeros(n)
        for i, m in enumerate(mean_global):
            overdraft[i] = self._overdraft_for(m)
        self.overdraft_limit = torch.tensor(overdraft, dtype=torch.float32, device=DEVICE)
        self.equilibrium = torch.clamp(torch.tensor(40 * mean_global, dtype=torch.float32,
                                                      device=DEVICE), min=1.0)

        target_counts = np.zeros(n)
        for j, action in enumerate(self.ACTIONS):
            sub = self.client_profiles_df[self.client_profiles_df["action"] == action]
            if not sub.empty:
                target_counts += self.rng.integers(
                    sub["min_tx_per_month"].min(), sub["max_tx_per_month"].max() + 1, size=n)
        self.target_total_count = torch.tensor(target_counts, dtype=torch.float32, device=DEVICE)
        total = self.target_total_count.sum()
        self.client_weight = self.target_total_count / total if total > 0 else torch.zeros(n, device=DEVICE)

    def _overdraft_for(self, mean_amount: float) -> float:
        for _, row in self.overdraft_df.iterrows():
            lo = -np.inf if str(row["mean_amount_min"]) == "-inf" else float(row["mean_amount_min"])
            hi = np.inf if str(row["mean_amount_max"]) == "inf" else float(row["mean_amount_max"])
            if lo <= mean_amount < hi:
                return float(row["overdraft_limit"])
        return 0.0

    def _build_step_target_tensors(self):
        by_step = self.agg_df.groupby("step")["count"].sum()
        arr = np.zeros(720)
        for s, c in by_step.items():
            if 0 <= int(s) < 720:
                arr[int(s)] = c
        self.step_target_count = torch.tensor(arr, dtype=torch.float32, device=DEVICE)

        step_avg = np.zeros((720, len(self.ACTIONS)))
        step_std = np.zeros((720, len(self.ACTIONS)))
        for j, action in enumerate(self.ACTIONS):
            sub = self.agg_df[self.agg_df["action"] == action]
            for _, row in sub.iterrows():
                s = int(row["step"])
                if 0 <= s < 720:
                    step_avg[s, j] = row["avg"]
                    step_std[s, j] = row["std"]
        self.step_avg = torch.tensor(step_avg, dtype=torch.float32, device=DEVICE)
        self.step_std = torch.tensor(step_std, dtype=torch.float32, device=DEVICE)

## 4. `TorchMoMTSimEngine` — moteur vectorisé

Boucle externe séquentielle sur les 720 steps (dépendance de solde inter-steps réelle), boucle interne vectorisée par slots de transaction sur tous les clients simultanément.

In [3]:
class TorchMoMTSimEngine:
    """n_clients clients + n_merchants marchands + n_banks banques + n_mules mules,
    tous indexés dans un même tenseur `balance` (layout : clients | merchants | banks | mules)."""

    def __init__(self, params: TorchParameters, n_clients: int, n_merchants: int,
                 n_banks: int, n_mules: int = 60, max_slots_per_step: int = 6, seed: int = 1000):
        self.params = params
        self.n_clients = n_clients
        self.n_merchants = n_merchants
        self.n_banks = n_banks
        self.n_mules = n_mules
        self.CLIENT_OFFSET = 0
        self.MERCHANT_OFFSET = n_clients
        self.BANK_OFFSET = n_clients + n_merchants
        self.MULE_OFFSET = n_clients + n_merchants + n_banks
        self.n_actors = n_clients + n_merchants + n_banks + n_mules
        self.max_slots = max_slots_per_step
        self.gen = torch.Generator(device=DEVICE).manual_seed(seed)

        self.balance = torch.zeros(self.n_actors, dtype=torch.float32, device=DEVICE)
        self.balance[:n_clients] = params.initial_balance
        self.balance[self.MULE_OFFSET:self.MULE_OFFSET + n_mules] = 0.0

        self.overdraft_limit = torch.zeros(self.n_actors, dtype=torch.float32, device=DEVICE)
        self.overdraft_limit[:n_clients] = params.overdraft_limit

        self.merchant_high_risk = torch.rand(n_merchants, generator=self.gen, device=DEVICE) < 0.90
        self.merchant_refund_proba = torch.rand(n_merchants, generator=self.gen, device=DEVICE) * 0.65 + 0.30

        self.in_prob = torch.full((n_clients,), 0.5, device=DEVICE)
        self.out_prob = torch.full((n_clients,), 0.5, device=DEVICE)

        self.history_buf = torch.full((n_clients, 100), -1, dtype=torch.int64, device=DEVICE)
        self.history_ptr = torch.zeros(n_clients, dtype=torch.int64, device=DEVICE)

        self.amount_hist = torch.zeros((n_clients, len(params.ACTIONS), 10),
                                        dtype=torch.float32, device=DEVICE)
        self.amount_hist_ptr = torch.zeros((n_clients, len(params.ACTIONS)),
                                            dtype=torch.int64, device=DEVICE)

        self.log_step = []
        self.log_action = []
        self.log_amount = []
        self.log_orig = []
        self.log_dest = []
        self.log_old_orig = []
        self.log_new_orig = []
        self.log_old_dest = []
        self.log_new_dest = []
        self.log_success = []
        self.log_is_fraud = []
        self.log_is_flagged = []
        self.log_scenario = []

    def _spring_probabilities(self):
        k = 1.0 / self.params.equilibrium
        client_balance = self.balance[:self.n_clients]
        spring_force = k * (self.params.equilibrium - client_balance)
        new_in = 0.5 * (1 + spring_force + (self.in_prob - self.out_prob))
        new_in = torch.clamp(new_in, 0.0, 1.0)
        return new_in, 1.0 - new_in

    def _draw_action_indices(self, in_prob: torch.Tensor) -> torch.Tensor:
        n = self.n_clients
        weights = self.params.action_available.float().clone()
        for j, action in enumerate(self.params.ACTIONS):
            if action in self.params.IN_ACTIONS:
                weights[:, j] *= in_prob
            else:
                weights[:, j] *= (1.0 - in_prob)
        weights = weights + 1e-8
        weights = weights / weights.sum(dim=1, keepdim=True)
        action_idx = torch.multinomial(weights, num_samples=1, generator=self.gen).squeeze(1)
        return action_idx

    def _draw_amounts(self, step: int, action_idx: torch.Tensor) -> torch.Tensor:
        n = self.n_clients
        rows = torch.arange(n, device=DEVICE)
        mu_client = self.params.mean_amount_tensor[rows, action_idx]
        std_client = self.params.std_amount_tensor[rows, action_idx]
        mu_step = self.params.step_avg[step, action_idx]
        std_step = self.params.step_std[step, action_idx]

        mu = (mu_client + mu_step) / 2
        sigma = torch.sqrt(std_client**2 + std_step**2) / 2
        sigma = torch.clamp(sigma, min=1e-3)

        amounts = torch.normal(mu, sigma, generator=self.gen)
        for _ in range(5):
            neg_mask = amounts <= 0
            if not neg_mask.any():
                break
            resample = torch.normal(mu, sigma, generator=self.gen)
            amounts = torch.where(neg_mask, resample, amounts)
        amounts = torch.clamp(amounts, min=1.0)
        return amounts

    def _draw_counterparties(self, action_idx: torch.Tensor) -> torch.Tensor:
        n = self.n_clients
        is_merchant_action = torch.zeros(n, dtype=torch.bool, device=DEVICE)
        for j, action in enumerate(self.params.ACTIONS):
            if action in ("CASH_IN", "PAYMENT"):
                is_merchant_action |= (action_idx == j)

        use_history = (torch.rand(n, generator=self.gen, device=DEVICE) < 0.90) & \
                      (self.history_buf[:, 0] >= 0)

        valid_counts = (self.history_buf >= 0).sum(dim=1).clamp(min=1)
        rand_slot = (torch.rand(n, generator=self.gen, device=DEVICE) * valid_counts.float()).long()
        hist_choice = self.history_buf[torch.arange(n, device=DEVICE), rand_slot]

        rand_merchant = torch.randint(self.n_clients, self.n_clients + self.n_merchants,
                                       (n,), generator=self.gen, device=DEVICE)
        rand_client = torch.randint(0, self.n_clients, (n,), generator=self.gen, device=DEVICE)
        rand_new = torch.where(is_merchant_action, rand_merchant, rand_client)

        dest = torch.where(use_history & (hist_choice >= 0), hist_choice, rand_new)
        self_tx = dest == torch.arange(n, device=DEVICE)
        dest = torch.where(self_tx, rand_new, dest)
        return dest

    def _update_history(self, orig_idx: torch.Tensor, dest_idx: torch.Tensor, is_new: torch.Tensor):
        remember_mask = is_new & (torch.rand(len(orig_idx), generator=self.gen, device=DEVICE) < 0.90)
        idxs = orig_idx[remember_mask]
        if len(idxs) == 0:
            return
        ptrs = self.history_ptr[idxs]
        self.history_buf[idxs, ptrs] = dest_idx[remember_mask]
        self.history_ptr[idxs] = (ptrs + 1) % 100

    def _run_step_slot(self, step: int, slot_mask: torch.Tensor):
        n = self.n_clients
        in_prob, _ = self._spring_probabilities()
        action_idx = self._draw_action_indices(in_prob)
        amounts = self._draw_amounts(step, action_idx)
        dest_idx = self._draw_counterparties(action_idx)

        orig_idx = torch.arange(n, device=DEVICE)
        can_pay = (self.balance[orig_idx] - amounts) >= self.overdraft_limit[orig_idx]
        active = slot_mask & can_pay

        old_orig = self.balance[orig_idx].clone()
        old_dest = self.balance[dest_idx].clone()

        withdraw_amounts = torch.where(active, amounts, torch.zeros_like(amounts))
        self.balance.index_add_(0, orig_idx, -withdraw_amounts)
        self.balance.index_add_(0, dest_idx, withdraw_amounts)

        already_known = (self.history_buf == dest_idx.unsqueeze(1)).any(dim=1)
        is_new = ~already_known
        self._update_history(orig_idx[active], dest_idx[active], is_new[active])

        active_np = active.cpu().numpy()
        if active_np.any():
            action_names = np.array(self.params.ACTIONS)[action_idx.cpu().numpy()]
            self.log_step.extend([step] * active_np.sum())
            self.log_action.extend(action_names[active_np].tolist())
            self.log_amount.extend(amounts[active].cpu().numpy().tolist())
            self.log_orig.extend(orig_idx[active].cpu().numpy().tolist())
            self.log_dest.extend(dest_idx[active].cpu().numpy().tolist())
            self.log_old_orig.extend(old_orig[active].cpu().numpy().tolist())
            self.log_new_orig.extend(self.balance[orig_idx][active].cpu().numpy().tolist())
            self.log_old_dest.extend(old_dest[active].cpu().numpy().tolist())
            self.log_new_dest.extend(self.balance[dest_idx][active].cpu().numpy().tolist())
            self.log_success.extend([True] * active_np.sum())
            self.log_is_fraud.extend([False] * active_np.sum())
            self.log_is_flagged.extend([False] * active_np.sum())
            self.log_scenario.extend([None] * active_np.sum())

    def log_transaction(self, step: int, action: str, amount: float,
                         orig_idx: int, dest_idx: int, old_orig: float, new_orig: float,
                         old_dest: float, new_dest: float, is_fraud: bool = True,
                         is_flagged: bool = False, scenario: str = None, success: bool = True):
        """Point d'entrée unique utilisé par les fraudeurs pour journaliser une transaction."""
        self.log_step.append(step)
        self.log_action.append(action)
        self.log_amount.append(float(amount))
        self.log_orig.append(int(orig_idx))
        self.log_dest.append(int(dest_idx))
        self.log_old_orig.append(float(old_orig))
        self.log_new_orig.append(float(new_orig))
        self.log_old_dest.append(float(old_dest))
        self.log_new_dest.append(float(new_dest))
        self.log_success.append(success)
        self.log_is_fraud.append(is_fraud)
        self.log_is_flagged.append(is_flagged)
        self.log_scenario.append(scenario)

    def transfer(self, orig_idx: int, dest_idx: int, amount: float) -> tuple:
        """Virement entre deux acteurs (indices globaux) sur le tenseur balance partagé."""
        old_orig = float(self.balance[orig_idx].item())
        old_dest = float(self.balance[dest_idx].item())
        can_pay = (old_orig - amount) >= float(self.overdraft_limit[orig_idx].item())
        if can_pay:
            self.balance[orig_idx] -= amount
            self.balance[dest_idx] += amount
        new_orig = float(self.balance[orig_idx].item())
        new_dest = float(self.balance[dest_idx].item())
        return old_orig, new_orig, old_dest, new_dest, can_pay

    def to_dataframe(self) -> pd.DataFrame:
        return pd.DataFrame({
            "step": self.log_step, "action": self.log_action, "amount": self.log_amount,
            "nameOrig": self.log_orig, "nameDest": self.log_dest,
            "oldBalanceOrig": self.log_old_orig, "newBalanceOrig": self.log_new_orig,
            "oldBalanceDest": self.log_old_dest, "newBalanceDest": self.log_new_dest,
            "isSuccessful": self.log_success, "isFraud": self.log_is_fraud,
            "isFlaggedFraud": self.log_is_flagged, "fraudScenario": self.log_scenario,
        })

## 5. `TorchFraudInjector` — les 5 scénarios (section 3.2 du mémoire)

Branché directement sur le tenseur `engine.balance` partagé.

**Corrections logiques apportées :**
- **ATO** : destinations = mules uniquement (jamais dans l'historique victime par construction)
- **Refund Fraud** : un agent fraudeur persistant ; compteur de cycles par marchand (non global) ;
  délai PAYMENT→REFUND Δt ~ U(1h, 48h) via file d'attente `_pending` ; 30 % transactions légitimes intercalées
- **Fake Credentials** : `is_flagged=False` (KYC n'a pas détecté — c'est tout le problème)
- **Split Deposit** : ε indépendant par fragment ; dernier fragment protégé contre valeur négative
- **Smurfing** : émetteur ≠ récepteur garanti ; `next_op_step` mis à jour même en cas de solde insuffisant
  (évite la boucle de retry) ; délai mule→récepteur Δt_mule ~ U(2h, 24h) via `_pending` ;
  renormalisation des x_i sans violer [0.70·S, 0.99·S]

In [4]:
class TorchFraudInjector:
    def __init__(self, engine: TorchMoMTSimEngine, params: TorchParameters,
                 fraud_probas: dict = None, seed: int = 1000):
        self.engine = engine
        self.params = params
        self.cfg = params.fraud_config
        self.gen = np.random.default_rng(seed)

        default_probas = {"ato": 0.02, "refund": 0.02, "fake_credentials": 0.005,
                           "split_deposit": 0.03, "smurfing_freq_mult": 1.0}
        self.probas = fraud_probas if fraud_probas is not None else default_probas

        n_c, n_m = engine.n_clients, engine.n_merchants
        self.client_ids = np.arange(n_c)
        self.merchant_ids = np.arange(engine.MERCHANT_OFFSET, engine.MERCHANT_OFFSET + n_m)
        self.mule_ids = np.arange(engine.MULE_OFFSET, engine.MULE_OFFSET + engine.n_mules)

        self._fake_cred_agents: dict = {}
        self._split_dep_agents = self.merchant_ids[engine.merchant_high_risk.cpu().numpy()]

        # File d'attente pour transactions différées (REFUND delay, mule→récepteur Smurfing)
        self._pending: list = []

        # Refund Fraud : un fraudeur persistant, liste ordonnée de marchands vulnérables,
        # compteur de cycles par marchand (non global)
        vuln_mask = engine.merchant_refund_proba.cpu().numpy() > self.cfg["refund"]["p_refund_threshold"]
        self._refund_vuln_list: list = list(self.merchant_ids[vuln_mask])
        self._refund_fraudster: int = int(self.gen.choice(self.client_ids))
        self._refund_merchant_idx: int = 0
        self._refund_cycle_count: int = 0

        # Smurfing : émetteur ≠ récepteur garanti
        n_networks = 5
        self.smurf_networks = []
        for _ in range(n_networks):
            emitter = int(self.gen.choice(self.client_ids))
            receiver_pool = self.client_ids[self.client_ids != emitter]
            receiver = int(self.gen.choice(receiver_pool))
            k = int(self.gen.integers(self.cfg["smurfing"]["n_mules_min"],
                                       self.cfg["smurfing"]["n_mules_max"] + 1))
            net_mules = self.gen.choice(self.mule_ids, size=min(k, len(self.mule_ids)), replace=False)
            n_conscious = int(len(net_mules) * self.cfg["smurfing"]["pct_conscious"])
            self.smurf_networks.append({
                "emitter": emitter, "receiver": receiver, "mules": net_mules,
                "conscious": set(net_mules[:n_conscious].tolist()),
                "next_op_step": int(self.gen.integers(0, 30 * 24)),
            })

    def _log(self, step, action, amount, orig, dest, scenario, flagged=False):
        old_o, new_o, old_d, new_d, success = self.engine.transfer(orig, dest, amount)
        self.engine.log_transaction(step, action, amount, orig, dest, old_o, new_o, old_d, new_d,
                                     is_fraud=True, is_flagged=flagged, scenario=scenario, success=success)
        return success

    def _log_legit(self, step, action, amount, orig, dest):
        """Transaction légitime de camouflage (is_fraud=False, is_flagged=False)."""
        old_o, new_o, old_d, new_d, success = self.engine.transfer(orig, dest, amount)
        self.engine.log_transaction(step, action, amount, orig, dest, old_o, new_o, old_d, new_d,
                                     is_fraud=False, is_flagged=False, scenario=None, success=success)
        return success

    def _flush_pending(self, current_step: int):
        """Exécute toutes les transactions différées dont le step planifié <= current_step."""
        remaining = []
        for entry in self._pending:
            sched_step, action, amount, orig, dest, scenario, flagged = entry
            if sched_step <= current_step:
                self._log(current_step, action, amount, orig, dest, scenario, flagged)
            else:
                remaining.append(entry)
        self._pending = remaining

    # 3.2.1 — ATO
    def _run_ato(self, step: int):
        c = self.cfg["ato"]
        victim = int(self.gen.choice(self.client_ids))
        B0 = float(self.engine.balance[victim].item())
        if B0 < c["B_min"]:
            return
        n = int(self.gen.integers(c["n_min"], c["n_max"] + 1))
        chosen_mules = self.gen.choice(self.mule_ids, size=min(n, len(self.mule_ids)), replace=False)
        remaining = B0
        for mule in chosen_mules:
            if remaining <= 0:
                break
            frac = self.gen.uniform(c["frag_min"], c["frag_max"])
            amount = min(remaining, frac * B0)
            if amount <= 1:
                continue
            self._log(step, "TRANSFER", amount, victim, int(mule), "ATO")
            remaining -= amount

    # 3.2.2 — Refund Fraud
    def _run_refund(self, step: int):
        c = self.cfg["refund"]
        if not self._refund_vuln_list or self._refund_merchant_idx >= len(self._refund_vuln_list):
            return

        # 30 % de transactions légitimes de camouflage (ratio_legit = 0.30)
        if self.gen.uniform() < c["ratio_legit"]:
            merchant = int(self.gen.choice(self.merchant_ids))
            amount = max(float(self.gen.normal(3325, 800)), 100.0)
            self._log_legit(step, "PAYMENT", amount, self._refund_fraudster, merchant)
            return

        merchant = self._refund_vuln_list[self._refund_merchant_idx]
        amount = max(float(self.gen.normal(3325, 800)), 100.0)

        # PAYMENT immédiat : fraudeur → marchand vulnérable
        self._log(step, "PAYMENT", amount, self._refund_fraudster, merchant, "REFUND")

        # REFUND différé : Δt ~ U(1h, 48h) — marchand rembourse le fraudeur (eq. 3.2.2)
        delay = int(self.gen.integers(c["delay_min_hours"], c["delay_max_hours"] + 1))
        self._pending.append((step + delay, "REFUND", amount, merchant, self._refund_fraudster, "REFUND", False))

        # Compteur par marchand (k_max cycles avant de passer au marchand suivant)
        self._refund_cycle_count += 1
        if self._refund_cycle_count >= c["k_max"]:
            self._refund_merchant_idx += 1
            self._refund_cycle_count = 0

    # 3.2.3 — Fake Credentials
    def _fake_credentials_step(self, step: int, allow_new: bool):
        c = self.cfg["fake_credentials"]
        if allow_new and len(self._fake_cred_agents) < 200:
            cid = int(self.gen.choice(self.client_ids))
            if cid not in self._fake_cred_agents:
                dormance_h = int(self.gen.integers(c["dormance_min_days"] * 24, c["dormance_max_days"] * 24))
                self._fake_cred_agents[cid] = {
                    "dormant_until": step + dormance_h, "n_leg_done": 0,
                    "n_leg_target": int(self.gen.integers(c["n_leg_min"], c["n_leg_max"] + 1)),
                    "activated": False,
                }

        for cid, state in list(self._fake_cred_agents.items()):
            if state["activated"]:
                continue
            if step < state["dormant_until"] and state["n_leg_done"] < state["n_leg_target"]:
                if self.gen.uniform() < 0.1:
                    merchant = int(self.gen.choice(self.merchant_ids))
                    amount = float(self.gen.uniform(500, c["m_leg_max"]))
                    old_o, new_o, old_d, new_d, success = self.engine.transfer(cid, merchant, amount)
                    self.engine.log_transaction(step, "PAYMENT", amount, cid, merchant, old_o, new_o,
                                                 old_d, new_d, is_fraud=False, scenario=None, success=success)
                    state["n_leg_done"] += 1
            elif step >= state["dormant_until"]:
                dest = int(self.gen.choice(self.client_ids))
                mean_amount = float(self.params.mean_amount_tensor[cid].mean().item())
                plafond = max(mean_amount * 10, 50000)
                amount = float(self.gen.uniform(c["m_exp_ratio_min"] * plafond, plafond))
                # is_flagged=False : le KYC n'a pas détecté l'usurpation — c'est l'essence du scénario
                self._log(step, "TRANSFER", amount, cid, dest, "FAKE_CRED", flagged=False)
                state["activated"] = True

    # 3.2.4 — Split Deposit
    def _optimal_fragmentation(self, total: float) -> list:
        grid = self.cfg["split_deposit"]["tariff_grid"]

        def commission(amount):
            c = 0.0
            for tier in grid:
                if amount >= tier["threshold"]:
                    c = tier["commission"]
            return c

        best_k, best_gain = 1, commission(total)
        for k in range(2, 6):
            frag = total / k
            gain_k = k * commission(frag)
            if gain_k > best_gain:
                best_gain, best_k = gain_k, k

        eps_max = self.cfg["split_deposit"]["epsilon_max"]
        base = total / best_k
        frags = []
        remaining = total
        for j in range(best_k - 1):
            eps_j = self.gen.uniform(0, eps_max)  # ε_j indépendant par fragment (eq. 3.2.4)
            frag = base + self.gen.uniform(-eps_j, eps_j)
            max_frag = remaining - (best_k - 1 - j) * 1.0
            frag = max(1.0, min(frag, max_frag))
            frags.append(frag)
            remaining -= frag
        frags.append(max(1.0, remaining))  # dernier fragment garanti positif, Σm_j = M conservée
        return frags

    def _run_split_deposit(self, step: int):
        if len(self._split_dep_agents) == 0:
            return
        agent = int(self.gen.choice(self._split_dep_agents))
        client = int(self.gen.choice(self.client_ids))
        total_deposit = float(self.gen.uniform(2000, 80000))
        if float(self.engine.balance[agent].item()) < total_deposit:
            return
        fragments = self._optimal_fragmentation(total_deposit)
        for frag in fragments:
            self._log(step, "CASH_IN", frag, agent, client, "SPLIT_DEP")

    # 3.2.5 — Smurfing (Zhdanova et al.)
    def _run_smurfing(self, step: int):
        c = self.cfg["smurfing"]
        smurf_mult = self.probas.get("smurfing_freq_mult", 1.0)
        for net in self.smurf_networks:
            if step < net["next_op_step"]:
                continue

            # next_op_step mis à jour AVANT la vérification du solde (évite busy-retry si solde insuffisant)
            interval = (c["operation_interval_days"] * 24) / max(smurf_mult, 1e-3)
            net["next_op_step"] = step + max(1, int(self.gen.normal(interval, 24)))

            total_X = float(self.gen.uniform(500000, 5000000))
            emitter_balance = float(self.engine.balance[net["emitter"]].item())
            if emitter_balance < total_X:
                continue

            s_seuil = c["S_seuil"]
            k = int(self.gen.integers(2, len(net["mules"]) + 1))
            chosen = self.gen.choice(net["mules"], size=min(k, len(net["mules"])), replace=False)
            # x_i ~ U(0.70·S, 0.99·S) indépendants, renormalisés pour respecter total_X
            raw = self.gen.uniform(0.70 * s_seuil, 0.99 * s_seuil, size=len(chosen))
            fractions = raw / raw.sum() * min(total_X, len(chosen) * 0.99 * s_seuil)

            for mule, x_i in zip(chosen, fractions):
                # Émetteur → mule (immédiat)
                self._log(step, "TRANSFER", float(x_i), net["emitter"], int(mule), "SMURFING")
                delta_i = self.gen.uniform(c["delta_min"], c["delta_max"])
                amount_out = float(x_i) * (1 - delta_i)
                # Mule → récepteur différé : Δt_mule ~ U(2h, 24h) (eq. 3.2.5)
                delay_mule = int(self.gen.integers(c["delay_mule_min_hours"], c["delay_mule_max_hours"] + 1))
                self._pending.append((step + delay_mule, "TRANSFER", amount_out,
                                       int(mule), net["receiver"], "SMURFING", False))

    def inject(self, step: int):
        # Traiter les transactions différées en premier (REFUNDs, mule→récepteur Smurfing)
        self._flush_pending(step)

        if self.gen.uniform() < self.probas["ato"]:
            self._run_ato(step)
        if self.gen.uniform() < self.probas["refund"]:
            self._run_refund(step)
        allow_new = self.gen.uniform() < self.probas["fake_credentials"]
        self._fake_credentials_step(step, allow_new)
        if self.gen.uniform() < self.probas["split_deposit"]:
            self._run_split_deposit(step)
        self._run_smurfing(step)

## 6. Calibration SSE / SPSA (section 3.1.3)

Minimisation de $\sum_c \sum_t (D_r(c,t) - D_s(c,t;\theta))^2$ par SPSA (PyTorch), sur une population réduite pour la vitesse, probas ensuite réutilisées à pleine échelle.

⚠️ Hypothèse méthodologique documentée : en l'absence de données réelles de fraude (section 1.7.1), $D_r$ (cible) est construite proportionnellement au volume légitime attendu par bin, équirépartie entre les 5 scénarios (section 3.1.1) — à signaler comme limite dans le mémoire.

In [5]:
class SSEFraudCalibrator:
    SCENARIOS = ["ato", "refund", "fake_credentials", "split_deposit", "smurfing"]
    SCENARIO_LABELS = {"ato": "ATO", "refund": "REFUND", "fake_credentials": "FAKE_CRED",
                        "split_deposit": "SPLIT_DEP", "smurfing": "SMURFING"}

    def __init__(self, param_dir: str, fraud_config_path: str, seed: int = 1000,
                 n_clients=500, n_merchants=100, n_banks=10, n_mules=30,
                 target_mid=0.23, n_steps=720, n_bins=30, n_seeds_per_eval=3,
                 max_slots_per_step=6):
        self.param_dir = param_dir
        self.fraud_config_path = fraud_config_path
        self.seed = seed
        self.n_clients = n_clients
        self.n_merchants = n_merchants
        self.n_banks = n_banks
        self.n_mules = n_mules
        self.target_mid = target_mid
        self.n_steps = n_steps
        self.n_bins = n_bins
        self.bin_size = n_steps // n_bins
        self.n_seeds_per_eval = n_seeds_per_eval
        self.max_slots_per_step = max_slots_per_step

        self._Dr = None
        self._legit_target_per_bin = None
        self._params_cache = None

    def _get_params(self) -> TorchParameters:
        if self._params_cache is None:
            self._params_cache = TorchParameters(
                self.param_dir, self.fraud_config_path, n_clients=self.n_clients, seed=self.seed)
        return self._params_cache

    def _build_target_distribution(self, params: TorchParameters) -> np.ndarray:
        legit_per_step = params.step_target_count.cpu().numpy()
        legit_per_bin = legit_per_step.reshape(self.n_bins, self.bin_size).sum(axis=1)
        self._legit_target_per_bin = legit_per_bin

        fraud_ratio = self.target_mid / (1 - self.target_mid)
        fraud_total_per_bin = legit_per_bin * fraud_ratio

        Dr = np.tile(fraud_total_per_bin / len(self.SCENARIOS), (len(self.SCENARIOS), 1))
        return Dr

    def _run_trial_binned(self, theta: np.ndarray, seed_offset: int) -> np.ndarray:
        probas = {
            "ato": float(theta[0]), "refund": float(theta[1]),
            "fake_credentials": float(theta[2]), "split_deposit": float(theta[3]),
            "smurfing_freq_mult": float(theta[4]),
        }

        params = self._get_params()
        engine = TorchMoMTSimEngine(
            params, n_clients=self.n_clients, n_merchants=self.n_merchants,
            n_banks=self.n_banks, n_mules=self.n_mules,
            max_slots_per_step=self.max_slots_per_step, seed=self.seed + seed_offset)
        injector = TorchFraudInjector(engine, params, fraud_probas=probas, seed=self.seed + seed_offset)

        for step in range(self.n_steps):
            n_tx_target = params.step_target_count[step]
            n_tx_per_client = torch.distributions.Binomial(
                total_count=n_tx_target.clamp(min=0),
                probs=params.client_weight.clamp(0, 1)
            ).sample()
            n_tx_per_client = torch.clamp(n_tx_per_client, max=engine.max_slots).long()

            for slot in range(engine.max_slots):
                slot_mask = n_tx_per_client > slot
                if slot_mask.any():
                    engine._run_step_slot(step, slot_mask)

            injector.inject(step)

        Ds = np.zeros((len(self.SCENARIOS), self.n_bins))
        if len(engine.log_step) == 0:
            return Ds

        steps_arr = np.array(engine.log_step)
        is_fraud_arr = np.array(engine.log_is_fraud)
        scenario_arr = np.array([s if s is not None else "" for s in engine.log_scenario])

        fraud_mask = is_fraud_arr
        if not fraud_mask.any():
            return Ds

        bins = np.clip(steps_arr[fraud_mask] // self.bin_size, 0, self.n_bins - 1)
        scenarios_f = scenario_arr[fraud_mask]

        for i, key in enumerate(self.SCENARIOS):
            label = self.SCENARIO_LABELS[key]
            sel = scenarios_f == label
            if sel.any():
                counts = np.bincount(bins[sel], minlength=self.n_bins)
                Ds[i, :] = counts[:self.n_bins]
        return Ds

    def _objective(self, theta: np.ndarray) -> float:
        theta = np.clip(theta, 1e-4, None)
        Ds_list = [self._run_trial_binned(theta, seed_offset=k)
                   for k in range(self.n_seeds_per_eval)]
        Ds_mean = np.mean(Ds_list, axis=0)
        sse = float(np.sum((self._Dr - Ds_mean) ** 2))
        return sse

    def calibrate(self, x0=None, maxiter=25, lr=0.05, spsa_c=0.02, verbose=True) -> dict:
        """SPSA (Simultaneous Perturbation Stochastic Approximation) : deux évaluations
        par itération suffisent à estimer un gradient approché, quel que soit le nombre
        de paramètres -- adapté à une simulation bruitée et non différentiable."""
        params_probe = self._get_params()
        self._Dr = self._build_target_distribution(params_probe)

        bounds_lo = torch.tensor([1e-4, 1e-4, 1e-4, 1e-4, 0.1])
        bounds_hi = torch.tensor([0.5, 0.5, 0.3, 0.5, 10.0])

        if x0 is None:
            theta = torch.tensor([0.02, 0.02, 0.005, 0.03, 1.0], dtype=torch.float32)
        else:
            theta = torch.tensor(x0, dtype=torch.float32)

        best_theta, best_sse = theta.clone(), float("inf")
        history = []

        for it in range(maxiter):
            delta = torch.tensor(np.random.choice([-1.0, 1.0], size=5), dtype=torch.float32)

            theta_plus = torch.clamp(theta + spsa_c * delta, bounds_lo, bounds_hi)
            theta_minus = torch.clamp(theta - spsa_c * delta, bounds_lo, bounds_hi)

            sse_plus = self._objective(theta_plus.numpy())
            sse_minus = self._objective(theta_minus.numpy())

            grad_hat = (sse_plus - sse_minus) / (2 * spsa_c * delta)
            grad_hat = torch.tensor(grad_hat, dtype=torch.float32)

            theta = theta - lr * grad_hat
            theta = torch.clamp(theta, bounds_lo, bounds_hi)

            sse_current = self._objective(theta.numpy())
            history.append({"iter": it, "sse": sse_current, "theta": theta.tolist()})
            if verbose:
                print(f"[iter {it}] theta={theta.tolist()} SSE={sse_current:,.1f}")

            if sse_current < best_sse:
                best_sse, best_theta = sse_current, theta.clone()

        theta_star = best_theta.numpy()
        probas_final = {
            "ato": float(theta_star[0]), "refund": float(theta_star[1]),
            "fake_credentials": float(theta_star[2]), "split_deposit": float(theta_star[3]),
            "smurfing_freq_mult": float(theta_star[4]),
        }
        return {"probas": probas_final, "sse_final": float(best_sse),
                "converged": best_sse < np.inf, "history": history}

## 7. Exécution du pipeline de génération

**Ne pas exécuter automatiquement** — étapes manuelles, à lancer cellule par cellule une fois les 6 CSV placés dans `./paramFiles/`.

In [7]:
# --- 7.1 Calibration (population réduite, ~150 runs complets -- peut être long) ---
calib = SSEFraudCalibrator(
    param_dir=PARAM_DIR, fraud_config_path=FRAUD_CONFIG_PATH,
    seed=SEED, n_clients=500, n_merchants=100, n_banks=10, n_mules=30,
    target_mid=0.23, n_steps=720, n_bins=30, n_seeds_per_eval=3)

result = calib.calibrate(maxiter=25)
print("\nProbas finales retenues :", result["probas"])
print("SSE final :", result["sse_final"])

with open("calibrated_probas.json", "w", encoding="utf-8") as f:
    json.dump(result["probas"], f, indent=2)

C:\Users\Miguel\AppData\Local\Temp\ipykernel_34564\3065021455.py:133: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  grad_hat = torch.tensor(grad_hat, dtype=torch.float32)


[iter 0] theta=[9.999999747378752e-05, 9.999999747378752e-05, 0.30000001192092896, 0.5, 10.0] SSE=6,824,025,973.9
[iter 1] theta=[0.5, 0.5, 0.30000001192092896, 0.5, 0.10000000149011612] SSE=6,820,014,239.1
[iter 2] theta=[0.5, 0.5, 0.30000001192092896, 9.999999747378752e-05, 0.10000000149011612] SSE=6,829,778,384.5
[iter 3] theta=[0.5, 0.5, 9.999999747378752e-05, 0.5, 10.0] SSE=6,820,396,686.5
[iter 4] theta=[9.999999747378752e-05, 0.5, 0.30000001192092896, 9.999999747378752e-05, 0.10000000149011612] SSE=6,829,566,133.1
[iter 5] theta=[0.5, 0.5, 9.999999747378752e-05, 0.5, 10.0] SSE=6,819,909,204.1
[iter 6] theta=[9.999999747378752e-05, 9.999999747378752e-05, 0.30000001192092896, 0.5, 0.10000000149011612] SSE=6,823,694,900.9
[iter 7] theta=[9.999999747378752e-05, 0.5, 0.30000001192092896, 0.5, 10.0] SSE=6,819,553,497.6
[iter 8] theta=[9.999999747378752e-05, 0.5, 0.30000001192092896, 9.999999747378752e-05, 10.0] SSE=6,829,757,300.5
[iter 9] theta=[0.5, 0.5, 0.30000001192092896, 0.5, 0.

In [8]:
# --- 7.2 Simulation complète (pleine échelle, probas calibrées) ---
N_CLIENTS, N_MERCHANTS, N_BANKS, N_MULES = 2000, 300, 20, 60
N_STEPS = 720

params = TorchParameters(PARAM_DIR, FRAUD_CONFIG_PATH, n_clients=N_CLIENTS, seed=SEED)
engine = TorchMoMTSimEngine(params, n_clients=N_CLIENTS, n_merchants=N_MERCHANTS,
                             n_banks=N_BANKS, n_mules=N_MULES, max_slots_per_step=6, seed=SEED)

fraud_probas = None
if os.path.exists("calibrated_probas.json"):
    with open("calibrated_probas.json", "r", encoding="utf-8") as f:
        fraud_probas = json.load(f)

injector = TorchFraudInjector(engine, params, fraud_probas=fraud_probas, seed=SEED)

for step in range(N_STEPS):
    n_tx_target = params.step_target_count[step]
    n_tx_per_client = torch.distributions.Binomial(
        total_count=n_tx_target.clamp(min=0),
        probs=params.client_weight.clamp(0, 1)
    ).sample()
    n_tx_per_client = torch.clamp(n_tx_per_client, max=engine.max_slots).long()

    for slot in range(engine.max_slots):
        slot_mask = n_tx_per_client > slot
        if slot_mask.any():
            engine._run_step_slot(step, slot_mask)

    injector.inject(step)

    if step % 50 == 0:
        print(f"step {step}/{N_STEPS} - {len(engine.log_step)} tx cumulees "
              f"({sum(engine.log_is_fraud)} frauduleuses)", flush=True)

df_raw = engine.to_dataframe()
df_raw.to_csv("rawLog_torch.csv", index=False)
fraud_rate = df_raw["isFraud"].mean()
print(f"\nTermine - {len(df_raw)} transactions, device={DEVICE}")
print(f"Taux de fraude global : {fraud_rate:.3f}")
print(df_raw.loc[df_raw["isFraud"], "fraudScenario"].value_counts(normalize=True))

step 0/720 - 0 tx cumulees (0 frauduleuses)
step 50/720 - 10376 tx cumulees (110 frauduleuses)
step 100/720 - 10814 tx cumulees (245 frauduleuses)
step 150/720 - 12188 tx cumulees (381 frauduleuses)
step 200/720 - 14456 tx cumulees (532 frauduleuses)
step 250/720 - 16310 tx cumulees (657 frauduleuses)
step 300/720 - 17840 tx cumulees (776 frauduleuses)
step 350/720 - 19330 tx cumulees (899 frauduleuses)
step 400/720 - 20055 tx cumulees (1019 frauduleuses)
step 450/720 - 20276 tx cumulees (1120 frauduleuses)
step 500/720 - 20482 tx cumulees (1254 frauduleuses)
step 550/720 - 20774 tx cumulees (1365 frauduleuses)
step 600/720 - 20970 tx cumulees (1487 frauduleuses)
step 650/720 - 21118 tx cumulees (1596 frauduleuses)
step 700/720 - 21307 tx cumulees (1694 frauduleuses)

Termine - 21380 transactions, device=cpu
Taux de fraude global : 0.081
fraudScenario
SPLIT_DEP    0.713956
REFUND       0.286044
Name: proportion, dtype: float64


## 8. Feature engineering (section 3.2.6 / 3.3) — 12 features vectorisées

Vectorisation par `searchsorted`/`cumsum` (O(n log n)), boucle Python résiduelle uniquement sur les comptes (pas sur les lignes).

In [9]:
FEATURES_12 = [
    "r1", "r2", "delta_B_orig", "delta_B_dest", "delta_commission_ratio",
    "var_agent_split", "rho_rupture", "rho_refund", "v1h", "flag_nuit",
    "rho_nouveau", "flag_anomalie",
]


class FeatureEngineer:
    """Calcule les features dérivées (section 3.2.6 / 3.3) à partir de rawLog."""

    def __init__(self, df: pd.DataFrame, eps: float = 1e-6):
        df = df.copy()
        # normalisation typage : le moteur torch indexe nameOrig/nameDest en entiers
        # (indices tensoriels globaux), pas des strings "C123" comme en NumPy pur.
        df["nameOrig"] = df["nameOrig"].astype(np.int64)
        df["nameDest"] = df["nameDest"].astype(np.int64)
        df["fraudScenario"] = df["fraudScenario"].fillna("NONE").astype(str)
        df["isFraud"] = df["isFraud"].astype(bool)
        self.df = df.sort_values(["step"]).reset_index(drop=True)
        self.eps = eps

    def compute_all(self) -> pd.DataFrame:
        df = self.df
        df = self._delta_balances(df)
        df = self._ratios_r1_r2(df)
        df = self._flag_anomalie(df)
        df = self._flag_nuit(df)
        df = self._velocity_1h(df)
        df = self._delta_commission_smurfing(df)
        df = self._var_agent_split_deposit(df)
        df = self._rho_rupture_fake_cred(df)
        df = self._rho_refund(df)
        df = self._rho_nouveau(df)
        self.df = df
        return df

    # eq. 3.8 / 3.9
    def _delta_balances(self, df: pd.DataFrame) -> pd.DataFrame:
        df["delta_B_orig"] = df["oldBalanceOrig"] - df["newBalanceOrig"]
        df["delta_B_dest"] = df["newBalanceDest"] - df["oldBalanceDest"]
        return df

    # eq. 3.10 / 3.11
    def _ratios_r1_r2(self, df: pd.DataFrame) -> pd.DataFrame:
        df["r1"] = df["amount"] / (df["oldBalanceOrig"] + self.eps)
        df["r2"] = df["amount"] / (df["newBalanceOrig"] + self.eps)
        df["r1_r2_product"] = df["r1"] * df["r2"]
        return df

    # eq. 3.12
    def _flag_anomalie(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.sort_values(["nameOrig", "action", "step"]).reset_index(drop=True)
        grp = df.groupby(["nameOrig", "action"])["amount"]
        rolling_mean = grp.transform(lambda s: s.shift(1).rolling(10, min_periods=2).mean())
        rolling_std = grp.transform(lambda s: s.shift(1).rolling(10, min_periods=2).std())
        threshold = rolling_mean + 2 * rolling_std
        df["flag_anomalie"] = (df["amount"] > threshold).fillna(False)
        return df.sort_values("step").reset_index(drop=True)

    # eq. 3.18
    def _flag_nuit(self, df: pd.DataFrame) -> pd.DataFrame:
        hour_of_day = df["step"] % 24
        df["flag_nuit"] = ((hour_of_day >= 22) | (hour_of_day < 6))
        return df

    # eq. 3.17
    def _velocity_1h(self, df: pd.DataFrame) -> pd.DataFrame:
        counts_per_step = df.groupby(["nameOrig", "step"]).size().rename("v1h_raw")
        df = df.merge(counts_per_step, on=["nameOrig", "step"], how="left")
        df["v1h"] = df["v1h_raw"]
        df = df.drop(columns=["v1h_raw"])
        return df

    # eq. 3.13, critere Zhdanova et al. 0 < delta <= 10%
    def _delta_commission_smurfing(self, df: pd.DataFrame) -> pd.DataFrame:
        df["delta_commission"] = np.nan
        df["delta_commission_ratio"] = np.nan
        df["is_mule_candidate"] = False

        transfers = df[df["action"] == "TRANSFER"].copy()
        if transfers.empty:
            return df

        in_tx = transfers[["nameDest", "step", "amount"]].rename(
            columns={"nameDest": "account", "amount": "amount_in", "step": "step_in"})
        out_tx = transfers[["nameOrig", "step", "amount"]].rename(
            columns={"nameOrig": "account", "amount": "amount_out", "step": "step_out"})

        in_tx = in_tx.sort_values(["account", "step_in"])
        out_tx = out_tx.sort_values(["account", "step_out"])

        merged = pd.merge_asof(
            out_tx.sort_values("step_out"), in_tx.sort_values("step_in"),
            left_on="step_out", right_on="step_in", by="account", direction="backward")
        merged["delta"] = merged["amount_in"] - merged["amount_out"]
        merged["delta_ratio"] = merged["delta"] / (merged["amount_in"] + self.eps)

        merged["is_mule_candidate"] = (merged["delta_ratio"] > 0) & (merged["delta_ratio"] <= 0.10)

        agg = merged.groupby("account").agg(
            delta_commission_ratio=("delta_ratio", "last"),
            is_mule_candidate=("is_mule_candidate", "any")).reset_index()

        df = df.merge(
            agg.rename(columns={"account": "nameOrig"}),
            on="nameOrig", how="left", suffixes=("", "_mule"))
        df["delta_commission_ratio"] = df["delta_commission_ratio_mule"].combine_first(
            df["delta_commission_ratio"])
        df["is_mule_candidate"] = df["is_mule_candidate_mule"].fillna(False)
        df = df.drop(columns=[c for c in df.columns if c.endswith("_mule")])
        return df

    # eq. 3.14
    def _var_agent_split_deposit(self, df: pd.DataFrame) -> pd.DataFrame:
        df["var_agent_split"] = np.nan
        df["k_fragments"] = 0

        cash_in = df[df["action"] == "CASH_IN"].copy()
        if cash_in.empty:
            return df

        grp = cash_in.groupby(["nameOrig", "nameDest", "step"])["amount"]
        stats = grp.agg(
            var_agent_split=lambda x: float(np.var(x)),  # ddof=0 (eq. 3.14), pas ddof=1 pandas
            k_fragments="count"
        ).reset_index()
        stats["var_agent_split"] = stats["var_agent_split"].fillna(0.0)

        df = df.merge(stats, on=["nameOrig", "nameDest", "step"], how="left", suffixes=("", "_new"))
        df["var_agent_split"] = df["var_agent_split_new"].combine_first(df["var_agent_split"])
        df["k_fragments"] = df["k_fragments_new"].fillna(0).astype(int)
        df = df.drop(columns=[c for c in df.columns if c.endswith("_new")])
        return df

    @staticmethod
    def _windowed_sum_by_group(steps: np.ndarray, values: np.ndarray,
                                lo_bounds: np.ndarray, hi_bounds: np.ndarray) -> np.ndarray:
        """Somme sur fenetre glissante via cumsum + searchsorted (O(n log n))."""
        cumsum = np.concatenate([[0.0], np.cumsum(values)])
        idx_lo = np.searchsorted(steps, lo_bounds, side="left")
        idx_hi = np.searchsorted(steps, hi_bounds, side="left")
        return cumsum[idx_hi] - cumsum[idx_lo]

    # eq. 3.15
    def _rho_rupture_fake_cred(self, df: pd.DataFrame) -> pd.DataFrame:
        window_steps = 30 * 24
        df = df.sort_values(["nameOrig", "step"]).reset_index(drop=True)

        mean_hist = np.full(len(df), np.nan)
        for _, idx in df.groupby("nameOrig").indices.items():
            idx = np.sort(idx)
            steps_g = df["step"].values[idx]
            amounts_g = df["amount"].values[idx]

            lo = steps_g - window_steps
            hi = steps_g

            sums = self._windowed_sum_by_group(steps_g, amounts_g, lo, hi)
            counts = self._windowed_sum_by_group(steps_g, np.ones_like(amounts_g), lo, hi)
            means = np.where(counts > 0, sums / np.maximum(counts, 1), np.nan)
            mean_hist[idx] = means

        df["mean_historique_30j"] = mean_hist
        df["rho_rupture"] = df["amount"] / (df["mean_historique_30j"].fillna(0) + self.eps)
        return df

    # eq. 3.16
    def _rho_refund(self, df: pd.DataFrame) -> pd.DataFrame:
        window_steps = 30 * 24
        df = df.sort_values(["nameOrig", "step"]).reset_index(drop=True)

        is_refund = (df["action"].values == "REFUND").astype(float)
        is_payment = (df["action"].values == "PAYMENT").astype(float)
        rho = np.zeros(len(df))

        for _, idx in df.groupby("nameOrig").indices.items():
            idx = np.sort(idx)
            steps_g = df["step"].values[idx]
            refund_g = is_refund[idx]
            payment_g = is_payment[idx]

            lo = steps_g - window_steps
            hi = steps_g + 1

            n_refund = self._windowed_sum_by_group(steps_g, refund_g, lo, hi)
            n_payment = self._windowed_sum_by_group(steps_g, payment_g, lo, hi)
            rho[idx] = n_refund / (n_payment + self.eps)

        df["rho_refund"] = rho
        return df

    # eq. 3.19
    def _rho_nouveau(self, df: pd.DataFrame) -> pd.DataFrame:
        window_hist = 90 * 24
        window_recent = 30 * 24
        df = df.sort_values(["nameOrig", "step"]).reset_index(drop=True)

        last_contact_step = df.groupby(["nameOrig", "nameDest"])["step"].shift(1)
        gap = df["step"] - last_contact_step
        is_new_dest = (last_contact_step.isna() | (gap > window_hist)).astype(float).values

        rho = np.zeros(len(df))
        for _, idx in df.groupby("nameOrig").indices.items():
            idx = np.sort(idx)
            steps_g = df["step"].values[idx]
            novelty_g = is_new_dest[idx]

            lo = steps_g - window_recent
            hi = steps_g + 1

            sums = self._windowed_sum_by_group(steps_g, novelty_g, lo, hi)
            counts = self._windowed_sum_by_group(steps_g, np.ones_like(novelty_g), lo, hi)
            rho[idx] = sums / np.maximum(counts, 1)

        df["rho_nouveau"] = rho
        return df

In [10]:
# --- Exécution du feature engineering ---
df_raw = pd.read_csv("rawLog_torch.csv")
engineer = FeatureEngineer(df_raw)
df_features = engineer.compute_all()
df_features.to_csv("featuresLog.csv", index=False)

df_features[["step", "action", "amount", "r1", "r2", "flag_anomalie",
             "flag_nuit", "v1h", "delta_commission_ratio",
             "var_agent_split", "rho_rupture", "rho_refund", "rho_nouveau"]].tail(20)

C:\Users\Miguel\AppData\Local\Temp\ipykernel_34564\3948945790.py:109: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["is_mule_candidate"] = df["is_mule_candidate_mule"].fillna(False)


,step,action,amount,r1,r2,flag_anomalie,flag_nuit,v1h,delta_commission_ratio,var_agent_split,rho_rupture,rho_refund,rho_nouveau
21360,175,CASH_IN,21978.373138,0.019150,0.019524,False,False,3,NaN,8532.818997,2.197837e+10,0.0,0.333333
21361,346,CASH_IN,5217.796834,0.004638,0.004660,False,False,4,NaN,55935.677117,2.380512e-01,0.0,0.285714
21362,346,CASH_IN,4895.770492,0.004372,0.004392,False,False,4,NaN,55935.677117,2.233595e-01,0.0,0.285714
21363,346,CASH_IN,4662.825277,0.004183,0.004200,False,False,4,NaN,55935.677117,2.127318e-01,0.0,0.285714
21364,346,CASH_IN,5227.509273,0.004625,0.004647,False,False,4,NaN,55935.677117,2.384944e-01,0.0,0.285714
21365,458,CASH_IN,7396.265145,0.006606,0.006650,False,True,5,NaN,11702.338020,6.037043e-01,0.0,0.250000
21366,458,CASH_IN,7126.539369,0.006284,0.006324,False,True,5,NaN,11702.338020,5.816885e-01,0.0,0.250000
21367,458,CASH_IN,7187.023029,0.006297,0.006337,False,True,5,NaN,11702.338020,5.866254e-01,0.0,0.250000
21368,458,CASH_IN,7255.781172,0.006318,0.006358,False,True,5,NaN,11702.338020,5.922376e-01,0.0,0.250000
21369,458,CASH_IN,7392.116475,0.006559,0.006603,False,True,5,NaN,11702.338020,6.033657e-01,0.0,0.250000


## 9. Validation topologique KAN (section 4.1)

Normalisation (eq. 4.1), Quick Decision Rule PCA (eq. 4.2/4.3), indice de Fisher (eq. 4.4), test KS (eq. 4.5), couverture de grille (eq. 4.6), règle de décision (eq. 4.7).

In [11]:
class TopologyValidator:
    def __init__(self, df_features: pd.DataFrame, features: list = None, eps: float = 1e-6):
        self.features = features or FEATURES_12
        self.eps = eps
        self.df = df_features.copy()
        for f in self.features:
            self.df[f] = self.df[f].astype(float)
        self.df[self.features] = self.df[self.features].fillna(0.0)
        self.y = self.df["isFraud"].astype(int).values

        self.mu = None
        self.sigma = None
        self.X_norm = None
        self.V = None
        self.singular_values = None
        self.Z = None
        self.report = {}

    # eq. 4.1
    def normalize(self) -> np.ndarray:
        X = self.df[self.features].values
        self.mu = X.mean(axis=0)
        self.sigma = X.std(axis=0)
        self.X_norm = (X - self.mu) / (self.sigma + self.eps)
        return self.X_norm

    # eq. 4.2 / 4.3
    def pca(self, k_max: int = None):
        if self.X_norm is None:
            self.normalize()
        Xc = self.X_norm - self.X_norm.mean(axis=0)
        U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
        self.V = Vt.T
        self.singular_values = S

        total_var = np.sum(S ** 2)
        ve2 = (S[0] ** 2 + S[1] ** 2) / total_var
        self.report["VE2"] = float(ve2)

        cum_var = np.cumsum(S ** 2) / total_var
        k80 = int(np.searchsorted(cum_var, 0.80) + 1)
        self.report["k_for_VE80"] = k80

        k = k_max or max(2, k80 if ve2 < 0.40 else 2)
        self.Z = Xc @ self.V[:, :k]
        self.report["k_used"] = k
        return self.Z

    # eq. 4.4
    def fisher_index(self, Z: np.ndarray = None) -> float:
        Z = Z if Z is not None else self.Z
        if Z is None:
            self.pca()
            Z = self.Z

        Z0 = Z[self.y == 0]
        Z1 = Z[self.y == 1]
        if len(Z0) == 0 or len(Z1) == 0:
            self.report["J_Fisher"] = float("nan")
            return float("nan")

        mu0, mu1 = Z0.mean(axis=0), Z1.mean(axis=0)
        S0 = (Z0 - mu0).T @ (Z0 - mu0)
        S1 = (Z1 - mu1).T @ (Z1 - mu1)
        Sw = S0 + S1

        Sw_inv = np.linalg.pinv(Sw)
        diff = (mu1 - mu0).reshape(-1, 1)
        numerator = float((diff.T @ Sw_inv @ diff).item())
        denominator = float(np.trace(Sw))
        j_fisher = numerator / (denominator + self.eps)

        self.report["J_Fisher"] = j_fisher
        return j_fisher

    # eq. 4.5 -- test KS vs loi normale de reference (features deja normalisees)
    @staticmethod
    def _ks_statistic_vs_normal(x: np.ndarray) -> float:
        x_sorted = np.sort(x)
        n = len(x_sorted)
        ecdf = np.arange(1, n + 1) / n
        cdf_normal = 0.5 * (1 + np.vectorize(math.erf)(x_sorted / np.sqrt(2)))
        d_stat = np.max(np.abs(ecdf - cdf_normal))
        return float(d_stat)

    def ks_per_feature(self) -> dict:
        if self.X_norm is None:
            self.normalize()
        ks_results = {}
        for j, feat in enumerate(self.features):
            col = self.X_norm[:, j]
            sample = col if len(col) <= 5000 else np.random.choice(col, 5000, replace=False)
            ks_results[feat] = self._ks_statistic_vs_normal(sample)
        self.report["ks_per_feature"] = ks_results
        self.report["ks_mean"] = float(np.mean(list(ks_results.values())))
        self.report["features_needing_transform"] = [f for f, d in ks_results.items() if d >= 0.15]
        return ks_results

    # eq. 4.6 -- grille supposee sur [-3, 3] (convention post z-score) -- hypothese a ajuster
    def grid_coverage(self, grid_min: float = -3.0, grid_max: float = 3.0) -> dict:
        if self.X_norm is None:
            self.normalize()
        coverage = {}
        for j, feat in enumerate(self.features):
            col = self.X_norm[:, j]
            x_min, x_max = col.min(), col.max()
            rho = (x_max - x_min) / (grid_max - grid_min + self.eps)
            coverage[feat] = float(rho)
        self.report["grid_coverage"] = coverage
        self.report["features_poor_coverage"] = [
            f for f, r in coverage.items() if not (0.8 <= r <= 1.0)]
        return coverage

    # eq. 4.7
    def decide(self) -> str:
        j_fisher = self.report.get("J_Fisher")
        ve2 = self.report.get("VE2")
        ks_mean = self.report.get("ks_mean")
        needs_transform = len(self.report.get("features_needing_transform", [])) > 0

        if j_fisher is None or ve2 is None or ks_mean is None:
            raise RuntimeError("Executer pca(), fisher_index() et ks_per_feature() avant decide().")

        if j_fisher > 1 and ve2 >= 0.40 and ks_mean < 0.15:
            decision = "KAN valide"
        elif j_fisher > 1 and needs_transform:
            decision = "Transformations requises"
        else:
            decision = "Architecture alternative"

        self.report["decision"] = decision
        return decision

    def run_full_validation(self) -> dict:
        self.normalize()
        self.pca()
        self.fisher_index()
        self.ks_per_feature()
        self.grid_coverage()
        self.decide()
        return self.report

    def apply_recommended_transforms(self) -> pd.DataFrame:
        """log(1+x) sur les features signalees par le test KS (valeurs brutes)."""
        to_transform = self.report.get("features_needing_transform", [])
        df_t = self.df.copy()
        for feat in to_transform:
            col = df_t[feat].values
            shifted = col - col.min() if col.min() < 0 else col
            df_t[feat] = np.log1p(shifted)
        return df_t

    def plot_pca_projection(self) -> go.Figure:
        if self.Z is None or self.Z.shape[1] < 2:
            self.pca(k_max=2)
        fig = go.Figure()
        for cls, color, label in [(0, "#636EFA", "Legitime"), (1, "#EF553B", "Fraude")]:
            mask = self.y == cls
            fig.add_trace(go.Scatter(
                x=self.Z[mask, 0], y=self.Z[mask, 1], mode="markers", name=label,
                marker=dict(size=4, color=color, opacity=0.5)))
        fig.update_layout(
            title=f"Quick Decision Rule - Projection PCA (VE2={self.report.get('VE2', 0):.3f}, "
                  f"J_Fisher={self.report.get('J_Fisher', 0):.3f})",
            xaxis_title="Composante principale 1", yaxis_title="Composante principale 2",
            template="plotly_white", height=550)
        return fig

    def plot_ks_summary(self) -> go.Figure:
        ks = self.report.get("ks_per_feature", {})
        if not ks:
            self.ks_per_feature()
            ks = self.report["ks_per_feature"]
        feats = list(ks.keys())
        vals = list(ks.values())
        colors = ["#EF553B" if v >= 0.15 else "#00CC96" for v in vals]
        fig = go.Figure(data=[go.Bar(x=feats, y=vals, marker_color=colors)])
        fig.add_hline(y=0.15, line_dash="dash", line_color="red",
                       annotation_text="seuil D_KS = 0.15")
        fig.update_layout(title="Statistique KS par feature (regularite des distributions)",
                           xaxis_title="Feature", yaxis_title="D_KS",
                           template="plotly_white", height=450)
        return fig

In [12]:
# --- Exécution de la validation topologique ---
df_features = pd.read_csv("featuresLog.csv")
validator = TopologyValidator(df_features)
report = validator.run_full_validation()

print("=== Rapport de validation topologique (section 4.1) ===")
print(f"VE2 (variance expliquee 2 composantes) : {report['VE2']:.4f}")
print(f"J_Fisher : {report['J_Fisher']:.4f}")
print(f"D_KS moyen : {report['ks_mean']:.4f}")
print(f"Features necessitant transformation : {report['features_needing_transform']}")
print(f"Features a couverture de grille insuffisante : {report['features_poor_coverage']}")
print(f"\nDECISION : {report['decision']}")

validator.plot_pca_projection().write_html("viz_pca_validation.html")
validator.plot_ks_summary().write_html("viz_ks_summary.html")

=== Rapport de validation topologique (section 4.1) ===
VE2 (variance expliquee 2 composantes) : 0.4055
J_Fisher : 0.0000
D_KS moyen : 0.4383
Features necessitant transformation : ['r1', 'r2', 'delta_B_orig', 'delta_B_dest', 'delta_commission_ratio', 'var_agent_split', 'rho_rupture', 'rho_refund', 'v1h', 'flag_nuit', 'rho_nouveau', 'flag_anomalie']
Features a couverture de grille insuffisante : ['r1', 'r2', 'delta_B_orig', 'delta_B_dest', 'delta_commission_ratio', 'var_agent_split', 'rho_rupture', 'rho_refund', 'v1h', 'flag_nuit', 'rho_nouveau', 'flag_anomalie']

DECISION : Architecture alternative


## 10. Visualisations Plotly

In [13]:
class MoMTSimVisualizer:
    def __init__(self, df_features: pd.DataFrame, df_target_agg: pd.DataFrame = None):
        self.df = df_features
        self.df_target = df_target_agg

    def plot_volume_per_action(self) -> go.Figure:
        counts = self.df.groupby(["step", "action"]).size().reset_index(name="count")
        fig = go.Figure()
        for action in counts["action"].unique():
            sub = counts[counts["action"] == action]
            fig.add_trace(go.Scatter(x=sub["step"], y=sub["count"], mode="lines",
                                      name=action, line=dict(width=1.5)))
        fig.update_layout(title="Volume de transactions par step et par action",
                           xaxis_title="Step (heure)", yaxis_title="Nombre de transactions",
                           template="plotly_white", height=450)
        return fig

    def plot_nrmse_comparison(self, action: str) -> go.Figure:
        if self.df_target is None:
            raise ValueError("df_target_agg requis pour cette visualisation")

        sim_counts = self.df[self.df["action"] == action].groupby("step").size()
        sim_counts = sim_counts.reindex(range(720), fill_value=0)

        target = self.df_target[self.df_target["action"] == action].set_index("step")["count"]
        target = target.reindex(range(720), fill_value=0)

        rmse = np.sqrt(np.mean((sim_counts.values - target.values) ** 2))
        span = target.values.max() - target.values.min()
        nrmse = rmse / span if span > 0 else np.nan

        fig = go.Figure()
        fig.add_trace(go.Scatter(x=list(range(720)), y=target.values, mode="lines",
                                  name="Cible (reel)", line=dict(color="red", width=1.5)))
        fig.add_trace(go.Scatter(x=list(range(720)), y=sim_counts.values, mode="lines",
                                  name="Simule", line=dict(color="blue", width=1.5, dash="dot")))
        fig.update_layout(
            title=f"Validation SSE - {action} (NRMSE = {nrmse:.4f})",
            xaxis_title="Step (heure)", yaxis_title="Nombre de transactions",
            template="plotly_white", height=450)
        return fig

    def plot_fraud_scenario_distribution(self) -> go.Figure:
        fraud_df = self.df[self.df["isFraud"]]
        counts = fraud_df["fraudScenario"].value_counts()
        fig = go.Figure(data=[go.Bar(x=counts.index, y=counts.values,
                                      marker_color=["#EF553B", "#636EFA", "#00CC96",
                                                    "#AB63FA", "#FFA15A"][:len(counts)])])
        fig.update_layout(title="Repartition des transactions frauduleuses par scenario",
                           xaxis_title="Scenario", yaxis_title="Nombre de transactions",
                           template="plotly_white", height=400)
        return fig

    def plot_fraud_timeline(self) -> go.Figure:
        fraud_df = self.df[self.df["isFraud"]]
        counts = fraud_df.groupby(["step", "fraudScenario"]).size().reset_index(name="count")

        fig = go.Figure()
        colors = {"ATO": "#EF553B", "REFUND": "#636EFA", "FAKE_CRED": "#00CC96",
                  "SPLIT_DEP": "#AB63FA", "SMURFING": "#FFA15A"}
        for scenario in counts["fraudScenario"].unique():
            sub = counts[counts["fraudScenario"] == scenario]
            fig.add_trace(go.Scatter(x=sub["step"], y=sub["count"], mode="markers+lines",
                                      name=scenario, marker=dict(size=4),
                                      line=dict(color=colors.get(scenario, "gray"), width=1)))
        fig.update_layout(title="Timeline des scenarios de fraude (720 steps = 30 jours)",
                           xaxis_title="Step (heure)", yaxis_title="Nombre de tx frauduleuses",
                           template="plotly_white", height=450)
        return fig

    def plot_r1_r2_scatter(self) -> go.Figure:
        sample = self.df.sample(min(5000, len(self.df)), random_state=42)
        fig = go.Figure()
        for is_fraud, color, label in [(False, "#636EFA", "Legitime"), (True, "#EF553B", "Fraude")]:
            sub = sample[sample["isFraud"] == is_fraud]
            fig.add_trace(go.Scatter(
                x=sub["r1"], y=sub["r2"], mode="markers", name=label,
                marker=dict(size=4, color=color, opacity=0.5)))
        fig.update_layout(title="r1 (montant/solde initial) vs r2 (montant/solde final)",
                           xaxis_title="r1", yaxis_title="r2", template="plotly_white", height=500)
        return fig

    def plot_feature_distributions(self, features: list = None) -> go.Figure:
        if features is None:
            features = ["r1", "r2", "rho_rupture", "rho_refund", "v1h", "rho_nouveau"]

        n = len(features)
        fig = make_subplots(rows=(n + 1) // 2, cols=2, subplot_titles=features)

        for i, feat in enumerate(features):
            row, col = i // 2 + 1, i % 2 + 1
            legit = self.df.loc[~self.df["isFraud"], feat].dropna()
            fraud = self.df.loc[self.df["isFraud"], feat].dropna()

            fig.add_trace(go.Histogram(x=legit, name="Legitime", marker_color="#636EFA",
                                        opacity=0.6, histnorm="probability density",
                                        showlegend=(i == 0)), row=row, col=col)
            fig.add_trace(go.Histogram(x=fraud, name="Fraude", marker_color="#EF553B",
                                        opacity=0.6, histnorm="probability density",
                                        showlegend=(i == 0)), row=row, col=col)

        fig.update_layout(title="Distributions des features cles - legitime vs fraude",
                           template="plotly_white", height=300 * ((n + 1) // 2), barmode="overlay")
        return fig

    def plot_smurfing_network_delta(self) -> go.Figure:
        smurf = self.df[self.df["fraudScenario"] == "SMURFING"]
        ratios = smurf["delta_commission_ratio"].dropna()

        fig = go.Figure()
        fig.add_trace(go.Histogram(x=ratios, nbinsx=50, marker_color="#FFA15A"))
        fig.add_vline(x=0.10, line_dash="dash", line_color="red",
                       annotation_text="seuil 10% (critere Zhdanova et al.)")
        fig.update_layout(title="Distribution de la commission mule observee (Smurfing)",
                           xaxis_title="delta_commission / montant recu", yaxis_title="Frequence",
                           template="plotly_white", height=400)
        return fig

In [14]:
# --- Génération des figures ---
df_target = None
try:
    df_target = pd.read_csv(f"{PARAM_DIR}/aggregatedTransactions.csv")
except FileNotFoundError:
    pass

viz = MoMTSimVisualizer(df_features, df_target)

figs = {
    "volume_per_action": viz.plot_volume_per_action(),
    "fraud_scenario_distribution": viz.plot_fraud_scenario_distribution(),
    "fraud_timeline": viz.plot_fraud_timeline(),
    "r1_r2_scatter": viz.plot_r1_r2_scatter(),
    "feature_distributions": viz.plot_feature_distributions(),
    "smurfing_delta": viz.plot_smurfing_network_delta(),
}

if df_target is not None:
    figs["nrmse_transfer"] = viz.plot_nrmse_comparison("TRANSFER")

for name, fig in figs.items():
    fig.write_html(f"viz_{name}.html")
    print(f"Genere : viz_{name}.html")

Genere : viz_volume_per_action.html
Genere : viz_fraud_scenario_distribution.html
Genere : viz_fraud_timeline.html
Genere : viz_r1_r2_scatter.html
Genere : viz_feature_distributions.html
Genere : viz_smurfing_delta.html
Genere : viz_nrmse_transfer.html


---
## Prochaine étape (hors ce notebook)

Chapitre 4 restant : architecture MKAN elle-même (cellule T-KAN à portes hybrides Gaussienne+Fourier, nœuds MultKAN, Grid Extension, fonction de perte régularisée L1+entropie) — à développer dans un notebook séparé une fois cette phase de préparation des données validée.